# OpenFold3 ddG Stand

Notebook UI for the panel-based WT-plus-19-mutants ddG stand.

Use it when you want to run one target over one or many mutable positions, keep resumable state in `state.sqlite`, and export a consensus ranking across ddG methods.

Workflow:
1. Edit the runtime cell if your OpenFold environment lives elsewhere.
2. Edit the user input cell with molecules, chain, and positions.
3. Run the preview cell to validate the target chain and planned mutation panels.
4. Run the launch cell to execute the stand.
5. Inspect the consensus table and exported CSV/JSON summary files.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd

NOTEBOOK_ROOT = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_ROOT.parent
HELPERS_DIR = NOTEBOOK_ROOT / 'helpers'
OPENFOLD_REPO_DIR = Path(
    os.environ.get('OPENFOLD_REPO_DIR', str(REPO_ROOT / 'openfold-3'))
).resolve()

for path in (HELPERS_DIR, OPENFOLD_REPO_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from of_notebook_lib import RuntimeConfig, preview_molecules, validate_molecules
from of_notebook_lib.query_builders import build_single_query_payload, normalize_molecules
from openfold3.panel_stand import PanelDdgStandRunner, PanelStandConfig
from openfold3.panel_summary import summarize_panel_state_db, write_panel_summary_outputs

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

CANONICAL_AA = 'ACDEFGHIKLMNPQRSTVWY'

def parse_positions(positions_text: str) -> tuple[int, ...]:
    values = []
    for token in positions_text.replace('\\n', ',').split(','):
        token = token.strip()
        if not token:
            continue
        values.append(int(token))
    if not values:
        raise ValueError('At least one position is required')
    return tuple(values)

def find_chain_sequence(molecules: list[dict], mutable_chain_id: str) -> str:
    for molecule in normalize_molecules(molecules):
        if mutable_chain_id in molecule['chain_ids']:
            sequence = molecule.get('sequence')
            if not sequence:
                raise ValueError(f'Chain {mutable_chain_id} does not have a sequence')
            return str(sequence)
    raise ValueError(f'Chain {mutable_chain_id} was not found in molecules')

def build_panel_preview(molecules: list[dict], mutable_chain_id: str, positions: tuple[int, ...]) -> pd.DataFrame:
    sequence = find_chain_sequence(molecules, mutable_chain_id)
    rows = []
    for position in positions:
        if position < 1 or position > len(sequence):
            raise ValueError(
                f'Position {position} is outside chain {mutable_chain_id} length {len(sequence)}'
            )
        wt_residue = sequence[position - 1]
        rows.append({
            'panel_id': f'{target_id}_{mutable_chain_id}_{wt_residue}{position}',
            'chain_id': mutable_chain_id,
            'position_1based': position,
            'wt_residue': wt_residue,
            'mutant_jobs': len(CANONICAL_AA) - 1,
        })
    return pd.DataFrame(rows)


## Runtime Setup

Adjust these paths only if your OpenFold3 environment or caches live elsewhere.


In [ ]:
runtime = RuntimeConfig(
    project_dir=REPO_ROOT,
    openfold_repo_dir=OPENFOLD_REPO_DIR,
    openfold_prefix=REPO_ROOT / '.venv',
    results_dir=REPO_ROOT / 'results',
    msa_cache_dir=REPO_ROOT / 'msa_cache' / 'colabfold_msas',
    triton_cache_dir=REPO_ROOT / '.runtime' / 'triton_cache',
    fixed_msa_tmp_dir=REPO_ROOT / '.runtime' / 'of3_colabfold_msas',
    use_fused_attention=False,
    use_deepspeed=False,
)

runtime_env = runtime.build_env()
for key in (
    'PATH',
    'LD_LIBRARY_PATH',
    'CUDA_HOME',
    'TRITON_CACHE_DIR',
    'PYTHONUNBUFFERED',
    'OPENFOLD_USE_FUSED_ATTENTION',
    'OPENFOLD_USE_DEEPSPEED',
):
    if key in runtime_env:
        os.environ[key] = runtime_env[key]
os.environ['OPENFOLD_PROJECT_DIR'] = str(runtime.project_dir)
os.environ['OPENFOLD_REPO_DIR'] = str(runtime.openfold_repo_dir)

runtime


## User Input

Edit only this cell for normal usage.


In [ ]:
experiment_name = 'ddg_panel_demo'
target_id = 'demo_target'

molecules = [
    {
        'molecule_type': 'protein',
        'chain_ids': ['A'],
        'sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
    },
    {
        'molecule_type': 'protein',
        'chain_ids': ['B'],
        'sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
    },
]

mutable_chain_id = 'B'
positions_text = '10, 15'

runner_yaml = None
inference_ckpt_path = None
inference_ckpt_name = None
msa_computation_settings_yaml = None
num_diffusion_samples = 1
num_model_seeds = 1
msa_panel_workers = 1
analysis_workers = 4
cleanup_intermediates = True

output_parent = runtime.results_dir / 'panel_ddg_stand'
summary_export_dirname = 'summary_exports'


## Input Preview

Validate molecules and inspect the planned mutation panels before launch.


In [ ]:
issues = validate_molecules(molecules)
if issues:
    for issue in issues:
        print('INPUT ISSUE:', issue)
else:
    print('Input looks valid.')

positions = parse_positions(positions_text)
sequence = find_chain_sequence(molecules, mutable_chain_id)
panel_preview_df = build_panel_preview(molecules, mutable_chain_id, positions)

display(preview_molecules(molecules))
display(panel_preview_df)
print('mutable_chain_id =', mutable_chain_id)
print('mutable_chain_length =', len(sequence))
print('panel_count =', len(panel_preview_df))
print('planned_mutant_jobs =', int(panel_preview_df['mutant_jobs'].sum()))


## Resolved Run Plan

This cell shows the exact paths and settings that will be used for the run.


In [ ]:
run_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
run_root = output_parent / f'{experiment_name}_{run_stamp}'
wt_query_path = run_root / 'wt_query.json'
wt_query_payload = build_single_query_payload(f'{target_id}_WT', molecules)

resolved_plan = {
    'run_root': str(run_root),
    'wt_query_path': str(wt_query_path),
    'target_id': target_id,
    'mutable_chain_id': mutable_chain_id,
    'positions': list(positions),
    'runner_yaml': None if runner_yaml is None else str(Path(runner_yaml)),
    'inference_ckpt_path': None if inference_ckpt_path is None else str(Path(inference_ckpt_path)),
    'inference_ckpt_name': inference_ckpt_name,
    'msa_computation_settings_yaml': None if msa_computation_settings_yaml is None else str(Path(msa_computation_settings_yaml)),
    'num_diffusion_samples': num_diffusion_samples,
    'num_model_seeds': num_model_seeds,
    'msa_panel_workers': msa_panel_workers,
    'analysis_workers': analysis_workers,
    'cleanup_intermediates': cleanup_intermediates,
}
print(json.dumps(resolved_plan, indent=2))


## Launch Experiment

This cell writes the WT query JSON, runs the panel stand, and exports the summary tables.


In [ ]:
run_root.mkdir(parents=True, exist_ok=True)
wt_query_path.write_text(
    json.dumps(wt_query_payload, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

panel_config = PanelStandConfig(
    target_id=target_id,
    wt_query_json=wt_query_path,
    output_root=run_root,
    mutable_chain_id=mutable_chain_id,
    positions=positions,
    runner_yaml=None if runner_yaml is None else Path(runner_yaml),
    inference_ckpt_path=None if inference_ckpt_path is None else Path(inference_ckpt_path),
    inference_ckpt_name=inference_ckpt_name,
    msa_computation_settings_yaml=(
        None if msa_computation_settings_yaml is None else Path(msa_computation_settings_yaml)
    ),
    num_diffusion_samples=num_diffusion_samples,
    num_model_seeds=num_model_seeds,
    msa_panel_workers=msa_panel_workers,
    analysis_workers=analysis_workers,
    cleanup_intermediates=cleanup_intermediates,
)

runner = PanelDdgStandRunner(panel_config)
try:
    run_payload = runner.run()
finally:
    runner.close()

panel_summary = summarize_panel_state_db(Path(run_payload['state_db']))
summary_outputs = write_panel_summary_outputs(
    panel_summary,
    Path(run_payload['output_root']) / summary_export_dirname,
)
panel_rows_df = pd.DataFrame([
    {
        'mutation': f"{row.chain_id}:{row.from_residue}{row.position_1based}{row.to_residue}",
        'predict_status': row.predict_status,
        'analysis_status': row.analysis_status,
        'consensus_z': row.consensus_z,
        'consensus_rank_desc': row.consensus_rank_desc,
        'rosetta_delta_vs_wt': row.rosetta_delta_vs_wt,
        **{f'score__{method}': row.method_scores.get(method) for method in panel_summary.methods},
    }
    for row in panel_summary.rows
])
top_consensus_df = pd.DataFrame(panel_summary.top_consensus)

print(json.dumps(run_payload, indent=2))
print(json.dumps({key: str(value) for key, value in summary_outputs.items()}, indent=2))


## Results

The first table shows the consensus shortlist. The second table contains the full per-mutation score matrix.


In [ ]:
print('target_id =', panel_summary.target_id)
print('total_jobs =', panel_summary.total_jobs)
print('analyzed_jobs =', panel_summary.analyzed_jobs)
print('fully_scored_jobs =', panel_summary.fully_scored_jobs)
print('methods =', panel_summary.methods)
display(top_consensus_df)
display(panel_rows_df.sort_values(['consensus_rank_desc', 'mutation'], na_position='last'))


## Resume / Re-Summarize

If a long run was interrupted, point `existing_run_root` at a previous output folder and rerun this cell to rebuild the exported summary tables without repeating inference.


In [ ]:
existing_run_root = None
# existing_run_root = run_root

if existing_run_root:
    existing_run_root = Path(existing_run_root)
    existing_summary = summarize_panel_state_db(existing_run_root / 'state.sqlite')
    existing_outputs = write_panel_summary_outputs(
        existing_summary,
        existing_run_root / summary_export_dirname,
    )
    print(json.dumps({key: str(value) for key, value in existing_outputs.items()}, indent=2))
    display(pd.DataFrame(existing_summary.top_consensus))
else:
    print('Set existing_run_root to an old run directory if you only want to rebuild summary exports.')
